###Load_input_to_raw

This notebook will read the reference csv file and load the data to the raw table

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("load_type", "")
dbutils.widgets.text("pipeline_run_id","")
dbutils.widgets.text("ingestion_date","")

In [0]:
load_type = dbutils.widgets.get("load_type")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

if load_type == "static":
  load = "ref/config"
else:
  load = "config"

In [0]:
%run /Workspace/e-commerce/Parameter_setup

##Parameter Setup
This notebook is used to connect to storage account 

Parameter setup completed


####Products

In [0]:
products_df = spark.read.csv(f"{inbound_path}/products.csv", header=True, inferSchema=True)
products_df = (products_df.withColumn("ingestion_date", to_timestamp(lit(ingestion_date)))
                          .withColumn("pipeline_run_id",lit(pipeline_run_id))
                )
products_df.createOrReplaceTempView("products")

In [0]:
spark.sql(f'''
INSERT INTO ecom_ref_raw.products
SELECT * FROM products
''').display()

####Product Category

In [0]:
product_category_df = spark.read.csv(f"{inbound_path}/product_category.csv", header=True, inferSchema=True)
product_category_df = (product_category_df.withColumn("ingestion_date", to_timestamp(lit(ingestion_date)))
                                          .withColumn("pipeline_run_id",lit(pipeline_run_id))
                    )
product_category_df.createOrReplaceTempView("product_category")

In [0]:
spark.sql(f'''
INSERT INTO ecom_ref_raw.product_category 
SELECT * FROM product_category
''').display()

####Geolocation

In [0]:
geolocation_df = spark.read.csv(f"{inbound_path}/geolocation.csv", header=True, inferSchema=True)
geolocation_df = (geolocation_df.withColumn("ingestion_date", to_timestamp(lit(ingestion_date)))
                                .withColumn("pipeline_run_id",lit(pipeline_run_id))
                )
geolocation_df.createOrReplaceTempView("geolocation")

In [0]:
spark.sql(f'''
INSERT INTO ecom_ref_raw.geolocation 
SELECT * FROM geolocation
''').display()